# 7. Wrap-up

Closes out the EDA series started in
[0_load_and_orient.ipynb](0_load_and_orient.ipynb). This notebook makes no
new findings — it consolidates the six steps behind it into one reference
for the preprocessing step: what's decided, what's flagged, and what's
still open.

| Step | Notebook | Answers |
|---|---|---|
| 0 | [Load & orient](0_load_and_orient.ipynb) | Is the data structurally sound — shapes, dtypes, duplicates, train/test schema match? |
| 1 | [Target variable](1_target_variable.ipynb) | What should the model predict — raw `SalePrice` or a transform? |
| 2 | [Missingness](2_missingness.ipynb) | Which `NaN`s mean "doesn't apply" vs. "wasn't recorded"? |
| 3 | [Univariate](3_univariate.ipynb) | What does each feature look like on its own, and does test match train? |
| 4 | [Bivariate](4_bivariate.ipynb) | How does each feature relate to the target? |
| 5 | [Multicollinearity](5_multicollinearity.ipynb) | How do features relate to *each other*? |
| 6 | [Outliers](6_outliers.ipynb) | Which rows don't fit — alone, against the target, or against every feature at once? |

**A convention held across all seven notebooks, worth stating once here
explicitly:** every one of them is *observational*. No `fillna`, transform,
encoding, drop, or row removal happens anywhere in `notebooks/eda/`. Each
notebook reloads `train.csv`/`test.csv` fresh and reports on it — the
findings below are inputs to a preprocessing step, not preprocessing
itself.

In [1]:
import numpy as np
import pandas as pd

# reload fresh — confirms nothing upstream in this series mutated the source files
df_train = pd.read_csv("../../data/train.csv")
df_test = pd.read_csv("../../data/test.csv")
df_train.shape, df_test.shape


((1460, 81), (1459, 80))

## 1. Target variable

**Decision: model on `log_sale_price = np.log1p(SalePrice)`.**

Raw `SalePrice` is heavily right-skewed (skew 1.88, kurtosis 6.54);
`log1p` brings both close to normal (0.12, 0.81) and roughly halves the
IQR-outlier count (61 → 28). A fitted Box-Cox λ (-0.077) landed close
enough to 0 (the log special case) that it wasn't worth the extra
parameter. Predictions get `expm1`-transformed back to dollars for
evaluation.

No missing values in `SalePrice` — no target imputation needed.

## 2. Missingness

Every `NaN` in this dataset was sorted into one of two buckets, verified
on both train and test:

| Bucket | Meaning | Handling |
|---|---|---|
| **Structural absence** | The feature genuinely doesn't apply (no pool, no alley, no basement, no garage, no fireplace, no masonry veneer) | `fillna("None")` / `fillna(0)` style — the value isn't missing, it's "doesn't exist" |
| **Real gap** | The feature applies but wasn't recorded | Needs an actual imputation strategy (median/mode/group-based) |

**Structural absence** (`PoolQC`, `MiscFeature`, `Alley`, `Fence`,
`FireplaceQu`, the garage group `Type`/`Qual`/`Finish`/`Cond`/`YrBlt`, the
basement group `Qual`/`Cond`/`Exposure`/`FinType1`/`FinType2`, and their
numeric companions) — confirmed structural on both splits, each with a
small number of named row-specific exceptions (a real feature present but
one sub-field unrecorded — e.g. `Id 2577` in test has a `GarageType` but
every other garage field, including the numeric ones, missing) that need
row-specific handling rather than the group default.

**Real gaps**: `LotFrontage` (17.7% train / 15.6% test — missing rate
tracks `LotConfig`, `CulDSac` highest), `Electrical` (1 row, `SBrkr`
dominant ~91%), and 7 test-only categorical fields (`MSZoning`,
`Utilities`, `Functional`, `SaleType`, `KitchenQual`, `Exterior1st`,
`Exterior2nd` — each with a strongly dominant category, mode-imputable).

## 3. Univariate

75% of numeric columns (25/33, temporal + presence/absence) are
skewed or zero-heavy enough to flag; 62% of ordinal columns (13/21) are
lopsided at <1% on their thinnest level; 72% of nominal columns (18/25)
have a rare category under 1%. Every observed ordinal level matched
`data_description.txt`'s documented order — no data-quality surprises
there.

**Train/test verification**: adversarial validation (`HistGradientBoostingClassifier`,
5-fold CV, shuffled) scored AUC ≈ 0.497 on all 33 `NUMERIC_COLS` — train
and test are statistically indistinguishable. The univariate flags found
in train generalize to test, not by assumption but by a direct check.

**Named exceptions**: `MSSubClass`'s category `150` appears in test but
never in train — needs an explicit unseen-category strategy at encoding
time (e.g. `OneHotEncoder(handle_unknown="ignore")`).

## 4. Bivariate (vs. target)

Combined numeric+ordinal Spearman ranking against `log_sale_price` is led
by `OverallQual`, `GrLivArea`, `GarageCars`. `Neighborhood` alone explains
over half the variance in nominal η² (0.57) — the strongest single
predictor of any type.

**Zero-inflated numeric columns** (11 continuous): 5 well-supported after
splitting has-X vs. none (`OpenPorchSF` +40%, `MasVnrArea` +34%,
`WoodDeckSF` +30%, `2ndFlrSF` +18%, `EnclosedPorch` -22% median price
lift), 3 too sparse to trust (`PoolArea` n=7, `LowQualFinSF` n=26,
`3SsnPorch` n=24 nonzero rows).

**Ordinal monotonicity**: 10/21 columns aren't monotonic against
documented level order; 4 are real, well-supported inversions
(`OverallCond`, `LotShape`, `LandSlope`, `BsmtFinType1`/`2` — likely
confounded with neighborhood/size, worth investigating before encoding);
8 are thin-level noise (n≤3, matching 3_univariate's lopsided flags).

**3_univariate's flat 5-column drop list, revised into individual
decisions** once each was checked against `log_sale_price`:

| Column | 3_univariate | 4_bivariate revision |
|---|---|---|
| `Utilities` | Drop candidate | Undecidable (minority n=1) — stands unchanged |
| `Street` | Drop candidate | Borderline — real-looking -30% direction, n=6 too thin to commit |
| `Condition2` | Drop candidate | Closest to undecidable — bucketed minority hides a bidirectional split at n=1-2/level |
| `RoofMatl` | Drop candidate | **Reversed to bucket/preserve** — minority +44.9%, driven by premium wood roofing (`WdShngl`/`WdShake`), not noise |
| `Heating` | Drop candidate | **Reversed to bucket** — minority consistently -21% to -56% below `GasA` across all 5 sub-levels |

## 5. Multicollinearity

No broad multicollinearity problem once two specific issues are set
aside:

- **Structural identities**: `TotalBsmtSF = BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF`
  and `GrLivArea = 1stFlrSF + 2ndFlrSF + LowQualFinSF`, exact for 100% of
  rows — not correlation, a data-generating identity. A linear model
  shouldn't get both a total and every part that sums to it.
- **VIF** (complete-case, 51 of 54 numeric+ordinal columns after excluding
  the identities, `Utilities`, and the two too-sparse ordinals `PoolQC`/`FireplaceQu`):
  only `BsmtFinSF1` clears 10 (11.1); 5 more sit in the 4-6 range
  (expected members of the basement/floor-area and garage/house-year
  groups already visible in the correlation heatmap).

**Other flagged pairs**: `GarageArea`/`GarageCars` (r=0.88, near-duplicate
size measures), `GarageYrBlt`/`YearBuilt` (r=0.83, a garage-age-relative-to-house-age
delta may carry more signal than either raw year), the quality-ordinal
cluster (`OverallQual`/`ExterQual`/`BsmtQual`/`KitchenQual`, mutual
r 0.6-0.73 — expected, not flagged as a problem).

**Nominal (Cramér's V)**: `MSSubClass`/`BldgType`/`HouseStyle` cluster
(0.85-0.90) is the one well-supported nominal redundancy —
`MSSubClass` is substantially a combination of the other two by
definition.

## 6. Outliers

Three methods triangulate on the same two rows:

| `Id` | `GrLivArea`>4000 | Residual (`OverallQual`+`GrLivArea`, \|z\|>3) | `IsolationForest` | Note |
|---|---|---|---|---|
| **524** | ✅ | ✅ (-6.9) | ✅ | `OverallQual`=10, `SalePrice`=$184,750, `SaleCondition`=Partial |
| **1299** | ✅ | ✅ (-8.8) | ✅ | `OverallQual`=10, `SalePrice`=$160,000, `SaleCondition`=Partial |
| 692 | ✅ | — | ✅ | Big *and* expensive ($755k) — fits the price trend |
| 1183 | ✅ | — | ✅ | Big *and* expensive ($745k) — fits the price trend |

`Id` 524 and 1299 are the strongest, most triangulated outlier candidates
— both new-construction (`Partial`) sales, both maximum quality, both
priced far below what their size predicts. 692/1183 are large and
expensive but consistent with the price trend, flagged only by the
unsupervised multivariate check — lower priority.

15 additional rows cleared \|residual z\|>3 with no single dominant
pattern (mostly small/low-quality homes underselling, one overselling) —
worth revisiting once the modeling feature set is larger than the
two-feature check used here.

## Consolidated action list for preprocessing

**Target**
- Model on `log_sale_price = np.log1p(SalePrice)`; invert with `expm1` for evaluation/submission.

**Missingness fills**
- Structural-absence columns → `fillna("None")` (categorical) / `fillna(0)` (numeric companions), with named row-specific exceptions handled individually (see [2_missingness.ipynb](2_missingness.ipynb) for the full per-row list).
- Real gaps (`LotFrontage`, `Electrical`, 7 test-only categoricals) → median/mode or group-based imputation.

**Transforms**
- Log/Box-Cox candidates (8 `SIZE_COLS`): `LotArea`, `LotFrontage`, `GrLivArea`, `TotalBsmtSF`, `1stFlrSF`, `BsmtFinSF1`, `BsmtUnfSF`, `GarageArea`.
- Has-X flag + transform on nonzero subset (5 well-supported zero-inflated): `OpenPorchSF`, `MasVnrArea`, `WoodDeckSF`, `2ndFlrSF`, `EnclosedPorch`.
- Presence-flag only, don't model magnitude (3 too-sparse zero-inflated): `PoolArea`, `LowQualFinSF`, `3SsnPorch`.

**Encoding**
- Ordinal: integer-encode on documented order; bucket the thin tail level for 8 columns (`ExterCond`, `Functional`, `HeatingQC`, `BsmtCond`, `GarageCond`, `OverallQual`, `PoolQC`, `GarageQual`); investigate a confound (neighborhood/lot size) before choosing an encoding for 4 columns with real inversions (`OverallCond`, `LotShape`, `LandSlope`, `BsmtFinType1`/`2`).
- Nominal: one-hot/dummy encode; bucket rare tails for `Neighborhood`, `Exterior1st`/`Exterior2nd`, `Condition1`, `RoofStyle`, `SaleType`, `Foundation`.
- Per-column drop/bucket decisions (revised from a flat 5-column list, see §4 table above): `Utilities` drop (undecidable), `Street`/`Condition2` borderline, `RoofMatl`/`Heating` bucket (not drop).
- `MSSubClass`'s unseen test-only category `150` needs `handle_unknown="ignore"` or an explicit fallback.

**Feature relationships**
- Don't feed both a structural-identity total and its parts to the same linear model: `TotalBsmtSF` vs. `BsmtFinSF1`+`BsmtFinSF2`+`BsmtUnfSF`; `GrLivArea` vs. `1stFlrSF`+`2ndFlrSF`+`LowQualFinSF`.
- Consider a `GarageYrBlt`-`YearBuilt` delta instead of two raw years.
- `MSSubClass` overlaps `BldgType`/`HouseStyle` — check whether a model needs all three.

**Outliers**
- `Id` 524, 1299: strongest candidates for exclusion (or a `SaleCondition`-aware adjustment) at training time.
- `Id` 692, 1183: lower-priority flag, large/expensive but on-trend.

**Boundary held across the whole series**: nothing above has been applied
to `df_train`/`df_test` in any of these seven notebooks — every item is a
reasoned recommendation for the preprocessing step to act on, not a
decision already made.

## Not yet decided

Deliberately left open — genuine tradeoffs that need a modeling context
(what algorithm, what metric behavior) to resolve, not more EDA:

- Exact imputation values/strategy per real-gap column (median vs.
  group-median, mode vs. `handle_unknown`).
- Final choice between a structural-identity total and its parts, or
  whether to engineer both into new features instead of picking one side.
- Whether `Id` 524/1299 get dropped outright, downweighted, or kept with a
  `SaleCondition`-derived flag.
- Bucket thresholds for the rare-tail ordinal/nominal columns (how thin is
  "too thin to encode alone").

This closes the EDA series — [0_load_and_orient.ipynb](0_load_and_orient.ipynb)
through here. Next step: preprocessing, applying the recommendations above
to build the actual training pipeline.